In [2]:
!pip install transformers pillow torch gradio

from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import gradio as gr

print("Loading BLIP model...")
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
print("Model loaded successfully!")

def caption_image(input_image):
    if input_image is None:
        return "Please upload an image."
    
    raw_image = Image.fromarray(input_image).convert("RGB")
    inputs = processor(raw_image, return_tensors="pt")
    out = model.generate(**inputs)
    caption = processor.decode(out[0], skip_special_tokens=True)
    return caption

demo = gr.Interface(
    fn=caption_image,
    inputs=gr.Image(label="Upload Any Image"),
    outputs=gr.Text(label="Generated AI Caption"),
    title="BLIP AI Image Captioning App"
)

# share=True creates a public web link inside Kaggle
demo.launch(share=True)

Loading BLIP model...


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Model loaded successfully!
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://3a700d1b5c93c779fe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Created dataset file at: .gradio/flagged/dataset1.csv


In [3]:
# 1. Install dependencies
!pip install -q transformers pillow torch torchvision torchaudio gradio

import requests
from PIL import Image
import gradio as gr
from transformers import (
    BlipProcessor, 
    BlipForConditionalGeneration, 
    BlipForQuestionAnswering
)

print("Loading models... (This takes ~30 seconds on Kaggle GPU)")

# Load Model 1: Base Captioning
proc_cap = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model_cap = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to("cuda")

# Load Model 2: Large Descriptive Q&A
proc_large = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
model_large = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large").to("cuda")

# Load Model 3: Dedicated Concise VQA
proc_vqa = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
model_vqa = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base").to("cuda")

print("All models loaded successfully!")

def analyze_image(input_image, question_text):
    if input_image is None:
        return "Please upload an image.", "Please upload an image.", "Please upload an image."
    
    raw_image = Image.fromarray(input_image).convert("RGB")
    
    # Task 1: Auto Caption
    inputs_cap = proc_cap(raw_image, return_tensors="pt").to("cuda")
    out_cap = model_cap.generate(**inputs_cap)
    caption_res = proc_cap.decode(out_cap[0], skip_special_tokens=True)
    
    # Task 2: Descriptive Q&A (Default question if empty)
    q_text = question_text if question_text.strip() != "" else "What is in the image?"
    inputs_large = proc_large(raw_image, q_text, return_tensors="pt").to("cuda")
    out_large = model_large.generate(**inputs_large)
    descriptive_res = proc_large.decode(out_large[0], skip_special_tokens=True)
    
    # Task 3: Direct VQA
    inputs_vqa = proc_vqa(raw_image, q_text, return_tensors="pt").to("cuda")
    out_vqa = model_vqa.generate(**inputs_vqa)
    vqa_res = proc_vqa.decode(out_vqa[0], skip_special_tokens=True)
    
    return caption_res, descriptive_res, vqa_res

# Create Gradio Interface
demo = gr.Interface(
    fn=analyze_image,
    inputs=[
        gr.Image(label="Upload Image"),
        gr.Textbox(label="Optional Question", placeholder="e.g., What animal is shown?")
    ],
    outputs=[
        gr.Textbox(label="1. Auto Caption (Base Model)"),
        gr.Textbox(label="2. Descriptive Answer (Large Model)"),
        gr.Textbox(label="3. Direct VQA Answer (VQA Base Model)")
    ],
    title="Complete BLIP Suite: Captioning & Visual Q&A"
)

# Launch public web link
demo.launch(share=True)

Loading models... (This takes ~30 seconds on Kaggle GPU)


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/788 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForQuestionAnswering LOAD REPORT from: Salesforce/blip-vqa-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_encoder.embeddings.position_ids      | UNEXPECTED |  | 
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


All models loaded successfully!
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://e6b521c4b8b0d848da.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 56, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Created dataset file at: .gradio/flagged/dataset2.csv
